# **Lab Report: Wing-Twist Optimisation Assignment**
<h3 align="RIGHT">Submitted by: Kalyana Raama Bheema<h3>

# **Gradient Descent: Wing-Twist Optimization**

Gradient descent is a method to find the minimum of a function $J(\boldsymbol{\alpha})$.\
The gradient $\nabla J$ points in the direction where $J$ increases the most, so we take a small step in the opposite direction:\
$$\boldsymbol{\alpha}_{k+1} = \boldsymbol{\alpha}_k - \eta\, \nabla J(\boldsymbol{\alpha}_k)$$\
$\eta$ is the learning rate (step size). We repeat this until the gradient is almost zero, because at the minimum $\nabla J = 0$.\
If $\eta$ is too small it takes many steps. If $\eta$ is too large the steps overshoot and $J$ blows up.

**Prandtl's lifting-line theory:**\
The spanwise coordinate is $y = -\frac{b}{2}\cos\theta$, with $\theta \in [0, \pi]$, and the circulation is written as a sine series:
$$\Gamma(\theta) = 2bV_\infty \sum_n A_n \sin(n\theta)$$\
For a symmetric wing only odd terms $n = 1, 3, 5, \dots$ are non-zero. The coefficients $A_n$ come from the monoplane equation at $N$ points on half the wing:
$$\sum_{j=1}^{N} A_{n_j} \sin(n_j\theta_i)\left[\frac{4b}{a_0 c(\theta_i)} + \frac{n_j}{\sin\theta_i}\right] = \alpha_i, \quad n_j = 2j-1, \quad \theta_i = \frac{i\pi}{2N}$$\
This is a linear system $M\mathbf{A} = \boldsymbol{\alpha}$. Once $\mathbf{A}$ is known:
$$C_L = \pi AR\, A_1, \qquad C_{D_i} = \pi AR \sum_j n_j A_{n_j}^2, \qquad e = \frac{C_L^2}{\pi AR\, C_{D_i}}$$

## **Problem Statement:**

*   **Wing twist optimization**: Twisting the wing changes the local angle of attack $\alpha(y)$, and with it the induced drag. We want the twist that gives the smallest induced drag for a required lift coefficient.

*   **Wing data**: Rectangular wing, $AR = 8$, $b = 1$, $c = b/AR$, $a_0 = 2\pi$ per radian, target $C_L^* = 0.5$.

*   **Design variables**: The $N = 12$ local angles $\boldsymbol{\alpha} = (\alpha_1, \dots, \alpha_N)$.

*   **Optimization problem**: The lift must equal the target, so a quadratic penalty is added to the drag:
$$\min_{\boldsymbol{\alpha} \in \mathbb{R}^N} J(\boldsymbol{\alpha}) = C_{D_i}(\boldsymbol{\alpha}) + \mu\,(C_L(\boldsymbol{\alpha}) - C_L^*)^2, \quad \mu = 10$$

*   **Elliptic loading (exact answer)**:
$$\alpha(\theta) = \frac{C_L}{\pi AR}\left[1 + \frac{4b}{a_0 c}\sin\theta\right]$$
This makes $A_n = 0$ for all $n \ge 3$, which gives $e = 1$.

In [1]:
# do not change line below
#from google.colab import files

# import required packages here
import numpy as np
import matplotlib.pyplot as plt

## **Task 1: Lifting-line function**

First a function that builds the matrix $M$:
$$M_{ij} = \sin(n_j\theta_i)\left[\frac{4b}{a_0 c} + \frac{n_j}{\sin\theta_i}\right]$$
Then a function that takes $\boldsymbol{\alpha}$, solves $M\mathbf{A} = \boldsymbol{\alpha}$ and returns $\mathbf{A}$, $C_L$, $C_{D_i}$ and $e$.

In [3]:
# wing data
AR = 8
b = 1
c = b / AR
a0 = 2 * np.pi
CL_target = 0.5

def matrix(N):
    theta = np.zeros(N)
    n = np.zeros(N)
    for i in range(N):
        theta[i] = (i + 1) * np.pi / (2 * N)
        n[i] = 2 * (i + 1) - 1
    M = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            M[i, j] = np.sin(n[j] * theta[i]) * (4 * b / (a0 * c) + n[j] / np.sin(theta[i]))
    return theta, n, M


def lifting_line(alpha):
    N = len(alpha)
    theta, n, M = matrix(N)
    A = np.linalg.solve(M, alpha)
    CL = np.pi * AR * A[0]
    CD = np.pi * AR * np.sum(n * A**2)
    e = CL**2 / (np.pi * AR * CD)
    return A, CL, CD, e

Now we do the validation for untwisted wing we just take the N values of $4, 8, 12, 16, 24$ and see where the slope converges and we know for untwisted wing all $\alpha_i$ are the same. If I put $\alpha_i = 1$ radian, then $C_L$ is directly the lift-curve slope $dC_L/d\alpha$.

In [4]:
Nl = [4, 8, 12, 16, 24]

print("  N   | dCL/dalpha (1/rad) |    e     |")
for N in Nl:
    alpha = np.ones(N)
    A, CL, CD, e = lifting_line(alpha)
    print(f"{N:<5} |      {CL:.6f}      | {e:.6f} |")

  N   | dCL/dalpha (1/rad) |    e     |
4     |      4.833329      | 0.939644 |
8     |      4.837454      | 0.936838 |
12    |      4.837656      | 0.936702 |
16    |      4.837689      | 0.936680 |
24    |      4.837701      | 0.936672 |


Here, from the table we can observe that the lift-curve slope goes to about 4.8377 per radian and $e$ goes to about 0.9367. From $N = 4$ to $N = 8$ the slope changes only in the 3rd decimal, and after $N = 12$ the change is in the 5th decimal. So the answer has converged and $N = 12$ is enough.\
The slope is less than $2\pi$ because the wing has finite span (the tip vortices reduce the angle the wing sees). $e < 1$ because its a rectangular wing.\
We can infer $N = 12$ from now on.

## **Task 2: Gradient descent from the untwisted wing**

**Gradient of $J$:**\
$\mathbf{A} = M^{-1}\boldsymbol{\alpha}$, so every $A_n$ is a linear function of $\boldsymbol{\alpha}$. Then
$$C_L = \pi AR\, A_1 \quad \Rightarrow \quad \frac{\partial C_L}{\partial \boldsymbol{\alpha}} = \pi AR \times (\text{first row of } M^{-1})$$
$$C_{D_i} = \pi AR \sum n_j A_{n_j}^2 \quad \Rightarrow \quad \frac{\partial C_{D_i}}{\partial \boldsymbol{\alpha}} = 2\pi AR\, (M^{-1})^T (n_j A_{n_j})$$
$$\nabla J = \frac{\partial C_{D_i}}{\partial \boldsymbol{\alpha}} + 2\mu\,(C_L - C_L^*)\frac{\partial C_L}{\partial \boldsymbol{\alpha}}$$

In [6]:
N = 12
theta, n, M = matrix(N)
Minv = np.linalg.inv(M) #doing the inverse of matrix M

def J(alpha, mu):
    A, CL, CD, e = lifting_line(alpha)
    return CD + mu * (CL - CL_target)**2


def grad_J(alpha, mu):
    A = Minv @ alpha
    CL = np.pi * AR * A[0]
    dCD = 2 * np.pi * AR * (Minv.T @ (n * A))
    dCL = np.pi * AR * Minv[0, :]
    return dCD + 2 * mu * (CL - CL_target) * dCL

Now, Let us define gradient descent function for our problem let us fix our tolerance as $10^-6$ and do the analysis for the untwisted wing that gives $C_L = C_L^*$:
$$\alpha_i = \frac{C_L^*}{dC_L/d\alpha} \quad \text{for all } i$$

In [7]:
def gradient_descent(grad, x0, lr, tol=1e-6):
    x = x0.copy()
    k = 0
    while True:
        g = grad(x)
        if np.linalg.norm(g) < tol:
            return x, k, "converged"
        if np.linalg.norm(g) > 1e6:
            return x, k, "diverged"
        x = x - lr * g
        k = k + 1
    return x, k

A, slope, CD, e = lifting_line(np.ones(N))
alpha0 = np.ones(N) * CL_target / slope

A0, CL0, CD0, e0 = lifting_line(alpha0)
print("untwisted angle =", np.degrees(alpha0[0]), "deg")
print("CL =", CL0, "  CDi =", CD0, "  e =", e0)

untwisted angle = 5.921853387014592 deg
CL = 0.49999999999999994   CDi = 0.010619370202042004   e = 0.9367018715790423


Running gradient descent with different fixed learning rates ($\mu = 10$):

In [15]:
mu = 10
lrl = [0.005, 0.01, 0.02, 0.03, 0.038, 0.039, 0.04, 0.05]

print("  lr    | iterations |      result      |      J       |")
for lr in lrl:
    alpha, k, status = gradient_descent(lambda a: grad_J(a, mu), alpha0, lr)
    print(f"{lr:<7} | {k:<10} | {status:<16} | {J(alpha, mu):.6e} |")

  lr    | iterations |      result      |      J       |
0.005   | 123656     | converged        | 9.907762e-03 |
0.01    | 61827      | converged        | 9.907762e-03 |
0.02    | 30912      | converged        | 9.907762e-03 |
0.03    | 20607      | converged        | 9.907762e-03 |
0.038   | 16268      | converged        | 9.907762e-03 |
0.039   | 16118      | converged        | 9.907762e-03 |
0.04    | 337        | diverged         | 9.911766e+09 |
0.05    | 38         | diverged         | 2.049274e+10 |


Here, first i checked with learning rates $ 0.005, 0.01, 0.02, 0.03, 0.04, 0.05 $ then i observed that max learning rate is in between 0.03 and 0.04. as the iterations are low around 0.04 so i tried 0.038, 0.039 there is the largest rate it converges.
**So the largest learning rate that still converges is about 0.039.**